# step A — RQ1 절벽 재현 (A1 생성)

**대응 RQ:** RQ1 — 선행 코드의 지침 위반이 이후 생성의 준수율을 낮추는가.

**무엇을 확인하나**
- 선행 12개(clone) 중 규약 준수(camelCase) 개수를 4/3/2/1/0으로 바꾸며 새 함수 3개를 순차 생성.
- 첫 함수 표기로 **준수율**, 턴1→2→3 연쇄로 **자기증폭**을 잰다.
- 기대: 준수 예시 ≥1이면 85~90% 유지, **0개에서 ~30% 급락(절벽)**. 첫 위반 시 자기증폭.

설계 문서: `docs/stepA/plan.md`, 프롬프트: `docs/stepA/prompt.md`.

> **메모리(무료 T4):** Qwen2.5-Coder-3B-Instruct fp16 ≈ 6.2GB로 T4(15GB)에 적재 가능.
> 100회 × 3턴 생성이라 실행에 대략 20~40분. 양자화 미사용(생성 실험이라 fp16).
> **재개 가능:** 조건마다 즉시 저장하고 이미 저장된 조건은 건너뛴다. 런타임이 끊겨도
> 셀 3·4·5를 다시 실행하면 남은 조건부터 이어서 한다.

In [ ]:
# 환경 설정 — 설치, GPU 확인, 시드 고정
!pip install -q transformers accelerate torch matplotlib pandas

import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (주의: 매우 느림)')

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('seed fixed:', SEED)

In [ ]:
# 저장소 클론 및 브랜치 체크아웃
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
!git fetch --quiet origin stepA/rq1-cliff
!git checkout stepA/rq1-cliff
!git pull --quiet origin stepA/rq1-cliff
!pip install -e . -q
import sys; sys.path.insert(0, 'src')

In [ ]:
# 조건 설정 — 이 실험이 쓰는 조건 축 값
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation)

MODEL = ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct', family='qwen', dtype='float16')
N_COMPLIANT = [4, 3, 2, 1, 0]   # 선행 12개 중 camel 준수 개수
SEEDS = list(range(20))          # 조건당 반복

def make(n, s):
    return Condition(
        model=MODEL,
        preceding=PrecedingCode(n_compliant=n, composition=Composition.CLONE),
        instruction=Instruction(form=InstructionForm.POSITIVE, target_notation=Notation.CAMEL),
        seed=s,
    )

conditions = [make(n, s) for n in N_COMPLIANT for s in SEEDS]

# 실행 전 예측 (결과와 함께 보존, CLAUDE.md §6)
PREDICTION = ('준수 예시 >=1이면 85~90% 유지, 0개에서 ~30%로 급락(절벽). '
              '첫 함수 위반 시 이후 함수도 위반으로 연쇄(자기증폭).')
print(len(conditions), '개 조건 =', len(N_COMPLIANT), 'x', len(SEEDS))

In [ ]:
# 실행 — 조건별 순차 생성 + 즉시 저장(재개 가능). 로직은 harness가 수행.
from harness import run, ResultRecord, save_result, result_path
from harness.model import load_model

handle = load_model(MODEL)
print('layers:', handle.num_layers, '| GQA:', handle.gqa_info())

new = skipped = 0
for c in conditions:
    if result_path(c, step='stepA').exists():   # 이미 한 조건 → 건너뜀(재개)
        skipped += 1
        continue
    out = run(c, handle=handle)
    save_result(ResultRecord(condition=out.condition, metrics=out.metrics,
                             step='stepA', rq='RQ1', prediction=PREDICTION))
    new += 1
    if new % 10 == 0:
        print(f'생성 {new} / 건너뜀 {skipped} / 총 {len(conditions)}')
print(f'완료: 새로 {new}, 건너뜀 {skipped}, 총 {len(conditions)}')

In [ ]:
# 결과 로드 — results/stepA/ 에 불변 저장된 이 실험 조건들을 모은다
from harness import result_path
from harness.results import load_result

records = [load_result(result_path(c, step='stepA')) for c in conditions]
print('로드:', len(records), '건 → results/stepA/')

In [ ]:
# 요약 — 준수율 절벽 곡선 + 자기증폭
import pandas as pd, matplotlib.pyplot as plt

rows = [{'n_compliant': r.condition.preceding.n_compliant,
         'compliant': r.metrics.extra['first_compliant'],
         'first_violated': r.metrics.extra['first_violated'],
         'subseq_viol': r.metrics.extra['subsequent_violation_rate']} for r in records]
df = pd.DataFrame(rows)

rate = df.groupby('n_compliant')['compliant'].mean().reindex(N_COMPLIANT)
print('준수율(첫 함수):'); print(rate.round(3))
amp = df[df.first_violated].groupby('n_compliant')['subseq_viol'].mean()
print('\n자기증폭(첫 위반 조건에서 이후 위반 비율):'); print(amp.round(3))

plt.figure(figsize=(5,3))
plt.plot([str(n) for n in N_COMPLIANT], rate.values, marker='o')
plt.xlabel('n_compliant (선행 준수 개수, 4->0)'); plt.ylabel('준수율(첫 함수)')
plt.title('RQ1 절벽'); plt.ylim(-0.02, 1.02); plt.grid(True, alpha=.3); plt.show()